# Statistical Inference in Regression — Solutions
### Applied Statistical Data Analysis — Prof. Dr. Kristyna Ters | MSc Finance | FHNW

---
> ⚠️ **Release to students only after the submission deadline.**


In [ ]:
!pip install yfinance statsmodels --quiet
import yfinance as yf, pandas as pd, numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan
import warnings; warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'white',
    'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':0.3})
YELLOW='#FDE70E'; RED='#C70101'; GREY='#4B4B4B'

# Shared data
data = yf.download(['AAPL','^GSPC'], start='2019-01-01', end='2024-12-31',
                   auto_adjust=True, progress=False)
ret = data['Close'].pct_change().dropna()
ret.columns = ['AAPL','SP500']
y = ret['AAPL']; X = sm.add_constant(ret['SP500'])
model = sm.OLS(y, X).fit()
print('✓ Data loaded.')

---
# Solution 1 — Identify Hypothesis Test Situations

| # | H₀ | H₁ | One/Two-sided? |
|---|----|----|----------------|
| a | β = 0 | β ≠ 0 | **Two-sided** (we don't know the direction) |
| b | α ≤ 0 | α > 0 | **One-sided** (we have a directional hypothesis) |
| c | β = 1 | β ≠ 1 | **Two-sided** (could be above or below market) |
| d | Sharpe ≤ 0 | Sharpe > 0 | **One-sided** (testing for positive performance) |

---
# Solution 2 — Manual t-test

In [ ]:
beta_hat = 0.85; se_beta = 0.12; n = 80; beta_0 = 0
t_stat = (beta_hat - beta_0) / se_beta
t_crit = stats.t.ppf(0.975, df=n-2)
p_val  = 2*(1-stats.t.cdf(abs(t_stat), df=n-2))
print(f't-stat = {t_stat:.3f}')
print(f't-critical (5%, two-sided) = ±{t_crit:.3f}')
print(f'p-value = {p_val:.6f}')
print(f'|t| = {abs(t_stat):.3f} > {t_crit:.3f} → REJECT H₀')
print(f'β is statistically significant at the 5% level')

---
# Solution 3 — Confidence Intervals

In [ ]:
beta_hat = 0.85; se_beta = 0.12; n = 80
t_95 = stats.t.ppf(0.975, df=n-2)
t_99 = stats.t.ppf(0.995, df=n-2)
ci_95 = (beta_hat - t_95*se_beta, beta_hat + t_95*se_beta)
ci_99 = (beta_hat - t_99*se_beta, beta_hat + t_99*se_beta)
print(f'95% CI: [{ci_95[0]:.3f}, {ci_95[1]:.3f}]')
print(f'99% CI: [{ci_99[0]:.3f}, {ci_99[1]:.3f}]')
print(f'β=1 in 95% CI: {ci_95[0] < 1 < ci_95[1]}')
print('\nAnswers:')
print('1. 99% CI is wider — higher confidence requires wider interval')
print('2. β=1 is NOT in 95% CI → we reject H₀:β=1 → stock is defensive (β<1)')
print('3. Doubling n halves the SE → CI narrows by factor √2 ≈ 1.41')

---
# Solution 4 — Full OLS Inference

In [ ]:
MY_STOCK = 'MSFT'
data_s = yf.download([MY_STOCK,'^GSPC'], start='2019-01-01', end='2024-12-31',
                     auto_adjust=True, progress=False)
ret_s = data_s['Close'].pct_change().dropna()
ret_s.columns = [MY_STOCK,'SP500']
y_s = ret_s[MY_STOCK]
X_s = sm.add_constant(ret_s['SP500'])
m_s = sm.OLS(y_s, X_s).fit()
ci_s = m_s.conf_int()
print(m_s.summary())
print(f'\n95% CI for beta: [{ci_s.iloc[1,0]:.4f}, {ci_s.iloc[1,1]:.4f}]')
print(f'Contains beta=1: {ci_s.iloc[1,0] < 1 < ci_s.iloc[1,1]}')

# Scatter
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(ret_s['SP500']*100, y_s*100, alpha=0.2, s=8, color='black')
x_l = np.linspace(ret_s['SP500'].min(), ret_s['SP500'].max(), 100)
ax.plot(x_l*100, (m_s.params[0]+m_s.params[1]*x_l)*100, color=YELLOW, lw=2.5)
ax.set_xlabel('S&P 500 (%)'); ax.set_ylabel(f'{MY_STOCK} (%)')
ax.set_title(f'CAPM Regression: {MY_STOCK}  β={m_s.params[1]:.3f}', fontweight='bold')
plt.tight_layout(); plt.show()

---
# Solution 5 — OLS Assumptions Check

In [ ]:
residuals = model.resid; fitted = model.fittedvalues

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].scatter(fitted*100, residuals*100, alpha=0.3, s=10, color='black')
axes[0].axhline(0, color=RED, ls='--', lw=1.5)
axes[0].set_title('Residuals vs. Fitted', fontweight='bold')
stats.probplot(residuals, dist='norm', plot=axes[1])
axes[1].get_lines()[0].set(color='black', markersize=3, alpha=0.5)
axes[1].get_lines()[1].set(color=YELLOW, lw=2.5)
axes[1].set_title('Q-Q Plot', fontweight='bold')
axes[2].plot(range(len(residuals)), residuals*100, color='black', lw=0.8)
axes[2].axhline(0, color=RED, ls='--')
axes[2].set_title('Residuals Over Time', fontweight='bold')
plt.tight_layout(); plt.show()

dw = durbin_watson(residuals)
bp_stat, bp_p, _, _ = het_breuschpagan(residuals, X)
print(f'Durbin-Watson: {dw:.3f} → {"Potential autocorrelation" if dw<1.5 or dw>2.5 else "OK"}')
print(f'Breusch-Pagan: p={bp_p:.4f} → {"Heteroskedasticity!" if bp_p<0.05 else "OK"}')

model_hac = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags':5})
print(f'\nSE comparison for beta:')
print(f'  OLS: {model.bse[1]:.5f}')
print(f'  HAC: {model_hac.bse[1]:.5f}')

---
# Solution 6 — p-value Interpretation

1. α: p=0.646 > 0.05 → **NOT** significant at 5%. Also not at 10% (0.646>0.10).
2. β: p=0.000 < 0.05 → **Significant**. 95% CI = 1.182 ± 1.96×0.083 = [1.019, 1.345].
3. Does CI contain β=1? Yes [1.019, 1.345] contains 1 → **Fail to reject H₀:β=1**.
4. p=0.000 is **NOT** exactly zero — it means p < 0.0005 (rounding). Always the smallest non-zero value the software can represent.

---
# Solution 7 — Simulate the t-distribution

In [ ]:
np.random.seed(42); N_SIM=1000; N_OBS=100; TRUE_BETA=0
t_stats = []
for _ in range(N_SIM):
    x = np.random.normal(0,1,N_OBS)
    y_sim = TRUE_BETA*x + np.random.normal(0,1,N_OBS)
    X_s = sm.add_constant(x)
    m = sm.OLS(y_sim, X_s).fit()
    t_stats.append(m.tvalues[1])
t_arr = np.array(t_stats)
t_crit = stats.t.ppf(0.975, df=N_OBS-2)
rej_rate = np.mean(np.abs(t_arr) > t_crit)
print(f'False rejection rate: {rej_rate:.3f} (theoretical: 0.050)')

fig, ax = plt.subplots(figsize=(8,5))
ax.hist(t_arr, bins=50, density=True, color='grey', alpha=0.7, label='Simulated t-stats')
x_t = np.linspace(-5,5,200)
ax.plot(x_t, stats.t.pdf(x_t, df=N_OBS-2), color='black', lw=2, label='Theoretical t-dist')
ax.axvline(t_crit, color=RED, ls='--'); ax.axvline(-t_crit, color=RED, ls='--')
ax.set_title('Simulated t-statistics under H₀: β=0', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()
print('Answers:')
print('1. With α=5%, we expect 5% of tests to incorrectly reject H₀ (Type I error rate)')
print('2. With TRUE_BETA=0.5, rejection rate >> 5% → this is POWER (correct rejections)')
print('3. Type I error = rejecting a true H₀. Controlled by our choice of α (5%)')

---
# Solution 8 — Heteroskedasticity

In [ ]:
np.random.seed(42); n=200
x = np.random.uniform(0, 3, n)
errors = np.random.normal(0, 0.2 + 0.5*x, n)
y_h = 0.5 + 1.2*x + errors
X_h = sm.add_constant(x)

m_ols = sm.OLS(y_h, X_h).fit()
m_hc3 = sm.OLS(y_h, X_h).fit(cov_type='HC3')

fig, axes = plt.subplots(1,2, figsize=(12,5))
axes[0].scatter(m_ols.fittedvalues, m_ols.resid, alpha=0.4, s=10)
axes[0].axhline(0, color=RED, ls='--')
axes[0].set_title('Residuals vs Fitted — Fan shape visible!', fontweight='bold')
bp_s, bp_p, _, _ = het_breuschpagan(m_ols.resid, X_h)
axes[0].text(0.05,0.95,f'BP test p={bp_p:.4f}\n→ Heteroskedasticity!',
             transform=axes[0].transAxes, fontsize=10,
             bbox=dict(facecolor='#FFEBEE', edgecolor='red', boxstyle='round,pad=0.3'))
axes[1].axis('off')
rows=[['Method','SE(β)','t-stat'],['OLS',f'{m_ols.bse[1]:.4f}',f'{m_ols.tvalues[1]:.3f}'],
      ['HC3',f'{m_hc3.bse[1]:.4f}',f'{m_hc3.tvalues[1]:.3f}']]
t5=axes[1].table(cellText=rows[1:],colLabels=rows[0],loc='center')
t5.auto_set_font_size(False); t5.set_fontsize(13); t5.scale(1.3,2.5)
for j in range(3): t5[0,j].set_facecolor('black'); t5[0,j].set_text_props(color='white')
axes[1].set_title('OLS vs HC3 Standard Errors', fontweight='bold', pad=15)
plt.tight_layout(); plt.show()
print('Answers:')
print('1. Fan shape: variance increases with fitted values → heteroskedasticity')
print('2. OLS SE is too small → inflated t-stats → spurious significance!')
print('3. HC3 SEs are larger → more conservative → more honest inference')

---
# Solution 9 — Sub-period Analysis

In [ ]:
data_f = yf.download(['AAPL','^GSPC'], start='2019-01-01', end='2024-12-31',
                     auto_adjust=True, progress=False)
ret_f = data_f['Close'].pct_change().dropna()
ret_f.columns=['AAPL','SP500']

results_sp = {}
for label, mask in [('COVID (2020)', (ret_f.index.year==2020)),
                    ('Post-COVID (2021-24)', (ret_f.index.year>=2021))]:
    r = ret_f.loc[mask]
    m = sm.OLS(r['AAPL'], sm.add_constant(r['SP500'])).fit()
    ci = m.conf_int().iloc[1]
    results_sp[label] = {'beta': m.params[1], 'se': m.bse[1],
                          'ci_lo': ci[0], 'ci_hi': ci[1], 'n': int(m.nobs)}
    print(f'{label}: β={m.params[1]:.4f}, SE={m.bse[1]:.4f}, 95%CI=[{ci[0]:.4f},{ci[1]:.4f}], n={int(m.nobs)}')

fig, ax = plt.subplots(figsize=(8,4))
for i, (label, res) in enumerate(results_sp.items()):
    ax.barh(i, res['ci_hi']-res['ci_lo'], left=res['ci_lo'],
            height=0.4, color=YELLOW, edgecolor='black')
    ax.plot([res['beta']]*2, [i-0.22,i+0.22], color=RED, lw=3)
ax.axvline(1.0, color='green', ls='--', lw=1.5)
ax.set_yticks(range(len(results_sp))); ax.set_yticklabels(results_sp.keys())
ax.set_title('Beta by Period — 95% CIs', fontweight='bold')
plt.tight_layout(); plt.show()

---
# Solution 10 — Risk-Return and Significance

In [ ]:
stocks = ['AAPL','MSFT','JPM','JNJ','TSLA']
data_m = yf.download(stocks+['^GSPC'], start='2019-01-01', end='2024-12-31',
                     auto_adjust=True, progress=False)
ret_m = data_m['Close'].pct_change().dropna()
sp500 = ret_m['^GSPC']

results_m = {}
for stock in stocks:
    m = sm.OLS(ret_m[stock], sm.add_constant(sp500)).fit()
    ci = m.conf_int().iloc[1]
    results_m[stock] = {'beta': m.params[1], 'ci_lo': ci[0], 'ci_hi': ci[1],
                         'reject_1': not(ci[0]<1<ci[1])}

fig, ax = plt.subplots(figsize=(10,5))
for i, (stock, res) in enumerate(results_m.items()):
    color = '#FFEBEE' if res['reject_1'] else '#E8F5E9'
    ax.barh(i, res['ci_hi']-res['ci_lo'], left=res['ci_lo'],
            height=0.5, color=YELLOW, edgecolor='black')
    ax.plot([res['beta']]*2, [i-0.28,i+0.28], color=RED, lw=3)
    marker = '✗' if res['reject_1'] else '✓'
    ax.text(res['ci_hi']+0.01, i, f'{res["beta"]:.2f} {marker}', va='center', fontsize=10)
ax.axvline(1.0, color='green', ls='--', lw=1.8, label='β=1')
ax.set_yticks(range(len(results_m))); ax.set_yticklabels(results_m.keys(), fontsize=11)
ax.set_xlabel('Beta (β)')
ax.set_title('Beta Estimates with 95% CIs\n✗=reject β=1, ✓=fail to reject', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

---
# Solution — Challenge: Bootstrap CIs

In [ ]:
np.random.seed(42); N_BOOT=5000
y_arr = y.values; X_arr = X.values
boot_betas = []
for _ in range(N_BOOT):
    idx = np.random.choice(len(y_arr), len(y_arr), replace=True)
    X_b, y_b = X_arr[idx], y_arr[idx]
    b = np.linalg.lstsq(X_b, y_b, rcond=None)[0]
    boot_betas.append(b[1])
boot_betas = np.array(boot_betas)
ci_boot = np.percentile(boot_betas, [2.5, 97.5])
ci_anal = model.conf_int().iloc[1].values
print(f'Analytical 95% CI: [{ci_anal[0]:.4f}, {ci_anal[1]:.4f}]')
print(f'Bootstrap  95% CI: [{ci_boot[0]:.4f}, {ci_boot[1]:.4f}]')
print('→ Very similar — analytical CI is valid here (assumptions hold)')

fig, ax = plt.subplots(figsize=(8,4))
ax.hist(boot_betas, bins=60, density=True, color='grey', alpha=0.7)
ax.axvline(ci_boot[0], color=RED, lw=2, ls='--')
ax.axvline(ci_boot[1], color=RED, lw=2, ls='--', label='Bootstrap 95% CI')
ax.axvline(model.params[1], color=YELLOW, lw=2.5, label=f'β̂={model.params[1]:.3f}')
ax.set_title('Bootstrap Distribution of β̂', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*

*⚠️ Release to students only after the submission deadline.*